In [1]:
import os
import gc
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import timm
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
# =========================
# 1. CONFIGURATION
# =========================
class CFG:
    SR = 32000
    DURATION = 5
    CHUNK_SIZE = SR * DURATION
    INPUT_DIR = Path("/kaggle/input/competitions/birdclef-2026/test_soundscapes/")
    MODEL_PATH = "/kaggle/input/models/gany24558/birdclef-apr-20-2026-gc/pytorch/composite-primary-labels/1/bird_model_best.pth"
    NUM_CLASSES = 234
    THRESHOLD = 0.00

    # Validation data paths
    TRAIN_CSV       = Path("/kaggle/input/competitions/birdclef-2026/train.csv")
    TAXONOMY_CSV    = Path("/kaggle/input/competitions/birdclef-2026/taxonomy.csv")
    TRAIN_AUDIO_DIR = Path("/kaggle/input/competitions/birdclef-2026/train_audio")
    PROCESSED_DIR   = Path("/kaggle/input/bird-clef-processed-features/processed_features_may_2")
    SOUNDSCAPE_CSV = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")
    SOUNDSCAPE_DIR = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")

In [3]:
# =========================
# 2. MODEL DEFINITION
# =========================
class BirdModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', num_classes=234):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=1)
        self.fc = nn.Linear(self.backbone.classifier.in_features, num_classes)
        self.backbone.classifier = nn.Identity()

    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return x

In [4]:
# =========================
# 3. INITIALIZATION
# =========================
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

model = BirdModel()
state_dict = torch.load(CFG.MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("Model loaded successfully")

Using device: cuda
Model loaded successfully


In [5]:
# =========================
# 4. SPECIES MAPPING
# =========================
taxonomy_df = pd.read_csv(CFG.TAXONOMY_CSV)
train_df    = pd.read_csv(CFG.TRAIN_CSV)

all_species_codes = sorted(taxonomy_df['primary_label'].astype(str).unique())
species_mapping   = {species: i for i, species in enumerate(all_species_codes)}
idx_to_species    = {v: k for k, v in species_mapping.items()}

print(f"Total species : {len(species_mapping)}")

# Training sample counts per species — useful for analysis
train_counts = train_df['primary_label'].astype(str).value_counts()
print(f"Train samples : {len(train_df)}")

Total species : 234
Train samples : 35549


In [6]:
# =========================
# 5. USE FULL TRAIN_DF FOR ANALYSIS
# =========================
# No need for train/val split here — we're just analyzing model performance
# Use the full train_df as our analysis set
val_df = train_df.copy()
#val_df = train_df.head(100).copy()

print(f'Analysis samples : {len(val_df)}')
print(f'Analysis species : {val_df["primary_label"].nunique()}')

Analysis samples : 35549
Analysis species : 206


In [7]:
# =========================
# 6. INFERENCE ON VALIDATION SET
# =========================
def get_spectrogram(audio_chunk):
    S = librosa.feature.melspectrogram(y=audio_chunk, sr=CFG.SR, n_mels=128, fmax=16000)
    return librosa.power_to_db(S, ref=np.max)


def predict_sample(filename):
    """Load raw audio, convert to spectrogram and run inference."""
    audio_path = CFG.TRAIN_AUDIO_DIR / filename
    if not audio_path.exists():
        return None
    
    try:
        y, _ = librosa.load(audio_path, sr=CFG.SR, duration=CFG.DURATION)
        # Pad if shorter than 5 seconds
        if len(y) < CFG.CHUNK_SIZE:
            y = np.pad(y, (0, CFG.CHUNK_SIZE - len(y)))
        spec   = get_spectrogram(y)
        tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(tensor)
            probs  = torch.sigmoid(output).cpu().numpy()[0]
        return probs
    except Exception as e:
        print(f'Error processing {filename}: {e}')
        return None



In [8]:
all_probs  = []
all_labels = []
skipped    = 0

for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc='Running inference'):
    probs = predict_sample(row['filename'])
    if probs is None:
        skipped += 1
        continue

    # Build one-hot label vector
    label_vec = np.zeros(CFG.NUM_CLASSES, dtype=np.float32)
    species   = str(row['primary_label'])
    if species in species_mapping:
        label_vec[species_mapping[species]] = 1.0

    """
    #TBD: Secondary
    secondary_labels = ast.literal_eval(row['secondary_labels'])
    
    for sec_label in secondary_labels:
        sec_label = str(sec_label)
    
        if sec_label in species_mapping:
            label_vec[species_mapping[sec_label]] = 0.5
    """

    all_probs.append(probs)
    all_labels.append(label_vec)

all_probs  = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

print(f'all_probs shape  : {all_probs.shape}')   # Expect (N, 234)
print(f'all_labels shape : {all_labels.shape}')  # Expect (N, 234)
print(f'Inference complete — {len(all_probs)} samples, {skipped} skipped')

Running inference: 100%|██████████| 35549/35549 [34:27<00:00, 17.19it/s]


all_probs shape  : (35549, 234)
all_labels shape : (35549, 234)
Inference complete — 35549 samples, 0 skipped


In [9]:
# =========================
# 7. PER-SPECIES AUC ANALYSIS
# =========================
species_auc = {}

for i, species in enumerate(all_species_codes):
    if all_labels[:, i].sum() == 0:
        continue  # skip species with no val samples
    try:
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
        species_auc[species] = auc
    except Exception:
        pass

auc_df = pd.DataFrame.from_dict(species_auc, orient='index', columns=['auc'])

# Enrich with training count and taxonomy info
auc_df['train_count'] = auc_df.index.map(train_counts).fillna(0).astype(int)
auc_df['class_name']  = auc_df.index.map(
    taxonomy_df.set_index('primary_label')['class_name']
)
auc_df['common_name'] = auc_df.index.map(
    taxonomy_df.set_index('primary_label')['common_name']
)
auc_df = auc_df.sort_values('auc')

print(f"Species evaluated : {len(auc_df)}")
print(f"Mean AUC          : {auc_df['auc'].mean():.4f}")
print(f"Median AUC        : {auc_df['auc'].median():.4f}")
print()
print("=" * 70)
print("WORST 20 PERFORMING SPECIES")
print("=" * 70)
print(auc_df.head(20)[['common_name', 'class_name', 'auc', 'train_count']].to_string())
print()
print("=" * 70)
print("BEST 20 PERFORMING SPECIES")
print("=" * 70)
print(auc_df.tail(20)[['common_name', 'class_name', 'auc', 'train_count']].to_string())

Species evaluated : 206
Mean AUC          : 0.9935
Median AUC        : 0.9945

WORST 20 PERFORMING SPECIES
                         common_name class_name       auc  train_count
nacnig1            Nacunda Nighthawk       Aves  0.967528           18
grekis                Great Kiskadee       Aves  0.968045          482
sobcac1             Solitary Cacique       Aves  0.968568          183
65377               Lesser Tree Frog   Amphibia  0.969879           46
whtdov             White-tipped Dove       Aves  0.971231          491
rufhor2               Rufous Hornero       Aves  0.975971          315
epaori4              Variable Oriole       Aves  0.977508          188
41970                         Jaguar   Mammalia  0.979388           21
bafcur1          Bare-faced Curassow       Aves  0.980300           51
chbmoc1     Chalk-browed Mockingbird       Aves  0.981380          270
bcwfin2  Black-capped Warbling Finch       Aves  0.981896           56
trokin             Tropical Kingbird     

In [10]:
# =========================
# 8. FAILURE MODE ANALYSIS
# =========================
print("=" * 70)
print("FAILURE PATTERN: Low AUC by Class")
print("=" * 70)
print(auc_df.groupby('class_name')['auc'].describe().round(4))

print()
print("=" * 70)
print("FAILURE PATTERN: Low AUC vs Training Count")
print("=" * 70)
# Bin by training count
bins   = [0, 10, 50, 100, 500, 10000]
labels = ['0-10', '11-50', '51-100', '101-500', '500+']
auc_df['count_bucket'] = pd.cut(auc_df['train_count'], bins=bins, labels=labels)
print(auc_df.groupby('count_bucket', observed=True)['auc'].describe().round(4))

FAILURE PATTERN: Low AUC by Class
            count    mean     std     min     25%     50%     75%  max
class_name                                                            
Amphibia     32.0  0.9982  0.0055  0.9699  0.9992  1.0000  1.0000  1.0
Aves        162.0  0.9923  0.0060  0.9675  0.9895  0.9931  0.9966  1.0
Insecta       3.0  0.9995  0.0009  0.9985  0.9992  1.0000  1.0000  1.0
Mammalia      8.0  0.9955  0.0084  0.9794  0.9962  1.0000  1.0000  1.0
Reptilia      1.0  1.0000     NaN  1.0000  1.0000  1.0000  1.0000  1.0

FAILURE PATTERN: Low AUC vs Training Count
              count    mean     std     min     25%     50%     75%     max
count_bucket                                                               
0-10           27.0  1.0000  0.0000  1.0000  1.0000  1.0000  1.0000  1.0000
11-50          25.0  0.9945  0.0093  0.9675  0.9933  0.9987  0.9998  1.0000
51-100         31.0  0.9931  0.0059  0.9803  0.9886  0.9946  0.9985  0.9999
101-500       123.0  0.9919  0.0056  0.9680  

In [11]:
# =========================
# 9. CONFUSION ANALYSIS
# What does the model predict for samples where it fails?
# =========================
print("=" * 70)
print("CONFUSION ANALYSIS — Worst 10 Species")
print("=" * 70)

worst_species = auc_df.head(10).index.tolist()

for species in worst_species:
    idx          = species_mapping[species]
    positive_mask = all_labels[:, idx] > 0.5

    if positive_mask.sum() == 0:
        continue

    model_preds   = all_probs[positive_mask, idx]
    correct_ratio = (model_preds > 0.5).mean()

    # What species does the model predict instead?
    mean_preds        = all_probs[positive_mask].mean(axis=0)
    mean_preds[idx]   = 0  # exclude the true species
    top3_confused_idx = mean_preds.argsort()[-3:][::-1]
    top3_confused     = [idx_to_species[i] for i in top3_confused_idx]
    top3_names        = [
        taxonomy_df[taxonomy_df['primary_label'] == s]['common_name'].values[0]
        if len(taxonomy_df[taxonomy_df['primary_label'] == s]) > 0 else s
        for s in top3_confused
    ]

    common = auc_df.loc[species, 'common_name']
    cls    = auc_df.loc[species, 'class_name']
    count  = auc_df.loc[species, 'train_count']

    print(f"\n{common} ({species}) [{cls}]")
    print(f"  AUC              : {auc_df.loc[species, 'auc']:.4f}")
    print(f"  Train count      : {count}")
    print(f"  Val positives    : {positive_mask.sum()}")
    print(f"  Correctly pred   : {correct_ratio:.1%}")
    print(f"  Mean pred score  : {model_preds.mean():.4f}")
    print(f"  Confused with    : {top3_names}")

CONFUSION ANALYSIS — Worst 10 Species

Nacunda Nighthawk (nacnig1) [Aves]
  AUC              : 0.9675
  Train count      : 18
  Val positives    : 18
  Correctly pred   : 83.3%
  Mean pred score  : 0.7746
  Confused with    : ['Southern Lapwing', 'Brown-chested Martin', 'Limpkin']

Great Kiskadee (grekis) [Aves]
  AUC              : 0.9680
  Train count      : 482
  Val positives    : 482
  Correctly pred   : 65.1%
  Mean pred score  : 0.6230
  Confused with    : ['Limpkin', 'Roadside Hawk', 'Rufous Hornero']

Solitary Cacique (sobcac1) [Aves]
  AUC              : 0.9686
  Train count      : 183
  Val positives    : 183
  Correctly pred   : 41.0%
  Mean pred score  : 0.3877
  Confused with    : ['Bluish-grey Saltator', 'Barred Antshrike', 'Great Antshrike']

Lesser Tree Frog (65377) [Amphibia]
  AUC              : 0.9699
  Train count      : 46
  Val positives    : 46
  Correctly pred   : 54.3%
  Mean pred score  : 0.5482
  Confused with    : ['Burrowing Owl', 'Tropical Kingbird', 'Cha

In [12]:
# =========================
# 10. SECONDARY LABEL ANALYSIS
# For the worst-performing species, inspect the most common
# secondary labels appearing in train.csv
# =========================

import ast
from collections import Counter

print("=" * 70)
print("SECONDARY LABEL ANALYSIS — Worst 10 Species")
print("=" * 70)

worst_species = auc_df.head(10).index.tolist()

for species in worst_species:

    # Filter rows for this primary species
    species_rows = train_df[
        train_df['primary_label'].astype(str) == species
    ]

    secondary_counter = Counter()

    # Collect all secondary labels
    for sec_labels_str in species_rows['secondary_labels']:

        try:
            sec_labels = ast.literal_eval(sec_labels_str)

            if isinstance(sec_labels, list):
                secondary_counter.update(
                    [str(s) for s in sec_labels]
                )

        except Exception:
            continue

    # Metadata
    common = auc_df.loc[species, 'common_name']
    cls    = auc_df.loc[species, 'class_name']
    auc    = auc_df.loc[species, 'auc']

    print(f"\n{common} ({species}) [{cls}]")
    print(f"  AUC              : {auc:.4f}")
    print(f"  Train samples    : {len(species_rows)}")

    if len(secondary_counter) == 0:
        print("  Secondary labels : None")
        continue

    # Top secondary labels
    print("  Top secondary labels:")

    for sec_species, count in secondary_counter.most_common(10):

        # Convert to common name if possible
        match = taxonomy_df[
            taxonomy_df['primary_label'].astype(str) == sec_species
        ]

        if len(match) > 0:
            sec_common = match['common_name'].values[0]
        else:
            sec_common = sec_species

        pct = count / len(species_rows)

        print(
            f"    - {sec_common} ({sec_species}) "
            f"-> {count} samples ({pct:.1%})"
        )

SECONDARY LABEL ANALYSIS — Worst 10 Species

Nacunda Nighthawk (nacnig1) [Aves]
  AUC              : 0.9675
  Train samples    : 18
  Top secondary labels:
    - Chalk-browed Mockingbird (chbmoc1) -> 1 samples (5.6%)
    - Southern Lapwing (soulap1) -> 1 samples (5.6%)
    - Ferruginous Pygmy Owl (fepowl) -> 1 samples (5.6%)

Great Kiskadee (grekis) [Aves]
  AUC              : 0.9680
  Train samples    : 482
  Top secondary labels:
    - Tropical Kingbird (trokin) -> 6 samples (1.2%)
    - Social Flycatcher (socfly1) -> 3 samples (0.6%)
    - House Sparrow (houspa) -> 2 samples (0.4%)
    - Rusty-margined Flycatcher (rumfly1) -> 2 samples (0.4%)
    - Palm Tanager (paltan1) -> 2 samples (0.4%)
    - Bananaquit (banana) -> 2 samples (0.4%)
    - White-tipped Dove (whtdov) -> 2 samples (0.4%)
    - Southern Beardless Tyrannulet (sobtyr1) -> 1 samples (0.2%)
    - Creamy-bellied Thrush (crbthr1) -> 1 samples (0.2%)
    - Pale-breasted Spinetail (pabspi1) -> 1 samples (0.2%)

Solitary Caci

In [13]:
# =========================
# 10. ACTIONABLE SUMMARY
# =========================
low_data_low_auc    = auc_df[(auc_df['train_count'] < 50) & (auc_df['auc'] < 0.7)]
enough_data_low_auc = auc_df[(auc_df['train_count'] >= 50) & (auc_df['auc'] < 0.7)]
insects_low_auc     = auc_df[(auc_df['class_name'] == 'Insecta') & (auc_df['auc'] < 0.7)]
amphibia_low_auc    = auc_df[(auc_df['class_name'] == 'Amphibia') & (auc_df['auc'] < 0.7)]

print("=" * 70)
print("ACTIONABLE SUMMARY")
print("=" * 70)
print(f"Low AUC (<0.7) + Low data (<50 samples) : {len(low_data_low_auc)} species")
print(f"  → Action: Need more training data (pseudo-labeling / soundscapes)")
print()
print(f"Low AUC (<0.7) + Enough data (≥50)      : {len(enough_data_low_auc)} species")
print(f"  → Action: Model isn't learning — check spectrogram quality or augmentation")
print()
print(f"Insects with low AUC                    : {len(insects_low_auc)} species")
print(f"  → Action: Insect sonotypes may need frequency-specific augmentation")
print()
print(f"Amphibians with low AUC                 : {len(amphibia_low_auc)} species")
print(f"  → Action: May benefit from fmax increase (frogs call at lower frequencies)")
print()
print(f"Overall mean AUC                        : {auc_df['auc'].mean():.4f}")

print()
print("Species needing more data:")
print(low_data_low_auc[['common_name', 'class_name', 'auc', 'train_count']]
      .sort_values('auc').to_string())

ACTIONABLE SUMMARY
Low AUC (<0.7) + Low data (<50 samples) : 0 species
  → Action: Need more training data (pseudo-labeling / soundscapes)

Low AUC (<0.7) + Enough data (≥50)      : 0 species
  → Action: Model isn't learning — check spectrogram quality or augmentation

Insects with low AUC                    : 0 species
  → Action: Insect sonotypes may need frequency-specific augmentation

Amphibians with low AUC                 : 0 species
  → Action: May benefit from fmax increase (frogs call at lower frequencies)

Overall mean AUC                        : 0.9935

Species needing more data:
Empty DataFrame
Columns: [common_name, class_name, auc, train_count]
Index: []


## Soundscape Analysis

In [14]:
def time_to_seconds(time_str):
    """Converts HH:MM:SS to integer seconds."""
    h, m, s = map(int, time_str.split(':'))
    return h * 3600 + m * 60 + s

# 1. Load the CSV
ss_df = pd.read_csv(CFG.SOUNDSCAPE_CSV)
initial_count = len(ss_df)

# 2. Update the parsing function
def parse_birds(label_str):
    if pd.isna(label_str) or label_str == '' or str(label_str).lower() == 'nocall':
        return []
    # Splitting by semicolon and cleaning whitespace
    return [b.strip() for b in str(label_str).split(';') if b.strip()]

In [15]:
# 3. Create initial bird lists and time columns
ss_df['bird_list'] = ss_df['primary_label'].apply(parse_birds)
ss_df['end_sec'] = ss_df['end'].apply(time_to_seconds)

# 4. Handle duplicates and merge labels for the same segment
# We group by filename and time window to merge species from multiple rows if they exist
ss_df = ss_df.groupby(['filename', 'start', 'end', 'end_sec']).agg({
    'bird_list': lambda x: sorted(list(set([bird for sublist in x for bird in sublist])))
}).reset_index()

# 5. Reporting
final_count = len(ss_df)
duplicates_removed = initial_count - final_count

print(f"Initial rows: {initial_count}")
print(f"Final segments (after merging duplicates): {final_count}")
print(f"Merged/Removed: {duplicates_removed} rows")

# Re-verify 'nocall' segments (segments with empty bird_list)
nocall_count = len(ss_df[ss_df['bird_list'].map(len) == 0])
print(f"Segments with calls: {final_count - nocall_count}")
print(f"Segments with 'nocall': {nocall_count}")

print(f"\nExample merged birds: {ss_df['bird_list'].iloc[0]}")

Initial rows: 1478
Final segments (after merging duplicates): 739
Merged/Removed: 739 rows
Segments with calls: 739
Segments with 'nocall': 0

Example merged birds: ['47158son13', '47158son17', '47158son22', '47158son23', '47158son25']


In [16]:
def predict_soundscape_window(filename, end_seconds):
    audio_path = CFG.SOUNDSCAPE_DIR / filename
    if not audio_path.exists():
        return None
    
    # Standard 5s window
    start_sec = max(0, end_seconds - 5)
    try:
        y, _ = librosa.load(audio_path, sr=CFG.SR, offset=start_sec, duration=CFG.DURATION)
        
        if len(y) < CFG.CHUNK_SIZE:
            y = np.pad(y, (0, CFG.CHUNK_SIZE - len(y)))
            
        spec = get_spectrogram(y)
        tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(tensor)
            probs = torch.sigmoid(output).cpu().numpy()[0]
        return probs
    except Exception as e:
        return None

ss_probs = []
ss_targets = []

# Use tqdm to track progress over your soundscapes
for _, row in tqdm(ss_df.iterrows(), total=len(ss_df), desc='Soundscape Inference'):
    probs = predict_soundscape_window(row['filename'], row['end_sec'])
    if probs is None: 
        continue

    # Build multi-hot target vector
    target_vec = np.zeros(CFG.NUM_CLASSES, dtype=np.float32)
    for bird in row['bird_list']:
        if bird in species_mapping:
            target_vec[species_mapping[bird]] = 1.0
            
    ss_probs.append(probs)
    ss_targets.append(target_vec)

ss_probs = np.vstack(ss_probs)
ss_targets = np.vstack(ss_targets)

Soundscape Inference: 100%|██████████| 739/739 [00:36<00:00, 20.05it/s]


In [17]:
from sklearn.metrics import average_precision_score

# 1. Calculate Global mAP
soundscape_map = average_precision_score(ss_targets, ss_probs, average='macro')
print(f"✅ FINAL CLEANED SOUNDSCAPE mAP: {soundscape_map:.4f}")

# 2. Per-Species Performance
species_results = []
for i, species in enumerate(all_species_codes):
    if ss_targets[:, i].sum() > 0:
        ap = average_precision_score(ss_targets[:, i], ss_probs[:, i])
        species_results.append({
            'species': species,
            'ap': ap,
            'count': int(ss_targets[:, i].sum())
        })

ap_df = pd.DataFrame(species_results).sort_values('ap')

print("\n--- CRITICAL FAILURES (Bottom 10 Species) ---")
print(ap_df.head(10).to_string(index=False))

print("\n--- BEST PERFORMERS (Top 10 Species) ---")
print(ap_df.tail(10).sort_values('ap', ascending=False).to_string(index=False))

✅ FINAL CLEANED SOUNDSCAPE mAP: 0.3036

--- CRITICAL FAILURES (Bottom 10 Species) ---
   species       ap  count
   rutjac1 0.333333      1
   ruther1 0.500000      1
   fusfly1 0.625000      3
   compot1 0.638889      3
   rufhor2 0.691667      4
47158son05 0.809524      3
   thlwre1 0.833333      2
    trsowl 0.852635     26
   purjay1 0.852857      5
47158son19 0.876923      5

--- BEST PERFORMERS (Top 10 Species) ---
species  ap  count
  65377 1.0      9
plcjay1 1.0      1
 limpki 1.0      3
nacnig1 1.0     11
  67252 1.0      2
bunibi1 1.0      2
  74113 1.0      2
 grekis 1.0      2
wfwduc1 1.0      1
strher2 1.0      2


In [18]:
# Use the results from the soundscape inference loop (ss_targets and ss_probs)
ss_species_auc = {}

for i, species in enumerate(all_species_codes):
    # Only evaluate species that are actually present in the soundscape labels
    if ss_targets[:, i].sum() == 0:
        continue 
        
    try:
        # Standard ROC-AUC for direct comparison with train.csv results
        auc = roc_auc_score(ss_targets[:, i], ss_probs[:, i])
        ss_species_auc[species] = auc
    except Exception:
        pass

ss_auc_df = pd.DataFrame.from_dict(ss_species_auc, orient='index', columns=['ss_auc'])

# Enrich with the same metadata as the train analysis
ss_auc_df['train_count'] = ss_auc_df.index.map(train_counts).fillna(0).astype(int)
ss_auc_df['class_name']  = ss_auc_df.index.map(
    taxonomy_df.set_index('primary_label')['class_name']
)
ss_auc_df['common_name'] = ss_auc_df.index.map(
    taxonomy_df.set_index('primary_label')['common_name']
)

# Crucial: Compare with the original AUC from train.csv if available
if 'auc' in auc_df.columns:
    ss_auc_df['train_auc'] = ss_auc_df.index.map(auc_df['auc'])
    ss_auc_df['auc_gap'] = ss_auc_df['train_auc'] - ss_auc_df['ss_auc']

ss_auc_df = ss_auc_df.sort_values('ss_auc')

print(f"Species in Soundscapes : {len(ss_auc_df)}")
print(f"Mean Soundscape AUC    : {ss_auc_df['ss_auc'].mean():.4f}")
print(f"Median Soundscape AUC  : {ss_auc_df['ss_auc'].median():.4f}")
print()
print("=" * 75)
print("WORST 20 PERFORMING SPECIES IN SOUNDSCAPES")
print("=" * 75)
cols = ['common_name', 'ss_auc', 'train_auc', 'auc_gap', 'train_count']
print(ss_auc_df.head(20)[cols].to_string())
print()
print("=" * 75)
print("BEST 20 PERFORMING SPECIES IN SOUNDSCAPES")
print("=" * 75)
print(ss_auc_df.tail(20)[cols].sort_values('ss_auc', ascending=False).to_string())

Species in Soundscapes : 75
Mean Soundscape AUC    : 0.9990
Median Soundscape AUC  : 0.9998

WORST 20 PERFORMING SPECIES IN SOUNDSCAPES
                                 common_name    ss_auc  train_auc   auc_gap  train_count
116570            Southern Spectacled Caiman  0.988875   1.000000  0.011125            1
undtin1                    Undulated Tinamou  0.990711   0.986269 -0.004442          204
trsowl                  Tropical Screech Owl  0.993365   0.991666 -0.001699          491
517063      Southern Orange-legged Leaf Frog  0.993760        NaN       NaN            0
compau                              Pauraque  0.994707   0.996553  0.001846          493
chacha1                     Chaco Chachalaca  0.994978   0.991008 -0.003970           99
fusfly1                   Fuscous Flycatcher  0.996830   0.994196 -0.002634          248
rutjac1                Rufous-tailed Jacamar  0.997290   0.991866 -0.005424          265
47158son10                 Insect sonotype10  0.997682        N

In [19]:
print("=" * 75)
print("SOUNDSCAPE FAILURE PATTERN: Performance by Class")
print("=" * 75)
# Analyzes if specific taxonomic groups (Insects vs Birds) are harder to detect in noise
print(ss_auc_df.groupby('class_name')['ss_auc'].describe().round(4))

print()
print("=" * 75)
print("SOUNDSCAPE FAILURE PATTERN: Performance vs Training Count")
print("=" * 75)
# Does 'More Data' in train.csv actually lead to better Soundscape performance?
bins   = [0, 10, 50, 100, 500, 10000]
labels = ['0-10', '11-50', '51-100', '101-500', '500+']
ss_auc_df['count_bucket'] = pd.cut(ss_auc_df['train_count'], bins=bins, labels=labels)

# We use observed=True to handle categories with no members in the soundscape subset
print(ss_auc_df.groupby('count_bucket', observed=True)['ss_auc'].describe().round(4))

print()
print("=" * 75)
print("DOMAIN SHIFT ANALYSIS: Average AUC Gap by Class")
print("=" * 75)
# This identifies classes that are 'Clean-Data Specialists' but 'Soundscape Failures'
if 'auc_gap' in ss_auc_df.columns:
    gap_analysis = ss_auc_df.groupby('class_name')['auc_gap'].agg(['mean', 'max', 'count']).sort_values('mean', ascending=False)
    print(gap_analysis.round(4))

SOUNDSCAPE FAILURE PATTERN: Performance by Class
            count    mean     std     min     25%     50%     75%     max
class_name                                                               
Amphibia     17.0  0.9993  0.0015  0.9938  0.9993  0.9995  1.0000  1.0000
Aves         28.0  0.9984  0.0023  0.9907  0.9979  0.9994  1.0000  1.0000
Insecta      25.0  0.9997  0.0007  0.9977  0.9998  1.0000  1.0000  1.0000
Mammalia      4.0  0.9995  0.0009  0.9982  0.9995  1.0000  1.0000  1.0000
Reptilia      1.0  0.9889     NaN  0.9889  0.9889  0.9889  0.9889  0.9889

SOUNDSCAPE FAILURE PATTERN: Performance vs Training Count
              count    mean     std     min     25%     50%     75%  max
count_bucket                                                            
0-10            8.0  0.9984  0.0039  0.9889  0.9993  0.9999  1.0000  1.0
11-50          11.0  0.9996  0.0006  0.9982  0.9994  0.9999  1.0000  1.0
51-100          7.0  0.9987  0.0017  0.9950  0.9988  0.9993  0.9995  1.0
101-500  

In [20]:
# Define thresholds for soundscape criticality
LOW_SS_AUC_THRESH = 0.70
HIGH_GAP_THRESH   = 0.15  # 15% drop from Train to Soundscape is critical

# Filtering
low_ss_auc_low_data = ss_auc_df[(ss_auc_df['ss_auc'] < LOW_SS_AUC_THRESH) & (ss_auc_df['train_count'] < 50)]
low_ss_auc_high_data = ss_auc_df[(ss_auc_df['ss_auc'] < LOW_SS_AUC_THRESH) & (ss_auc_df['train_count'] >= 50)]

# Domain Shift: Species the model knows well in clean audio but loses in noise
if 'auc_gap' in ss_auc_df.columns:
    high_gap_species = ss_auc_df[ss_auc_df['auc_gap'] > HIGH_GAP_THRESH]
else:
    high_gap_species = pd.DataFrame()

print("=" * 75)
print("SOUNDSCAPE ACTIONABLE SUMMARY")
print("=" * 75)

print(f"Low SS-AUC (<{LOW_SS_AUC_THRESH}) + Low data (<50)      : {len(low_ss_auc_low_data)} species")
print(f"  → Action: High priority for Pseudo-labeling. Model needs field context.")

print(f"\nLow SS-AUC (<{LOW_SS_AUC_THRESH}) + Enough data (≥50)   : {len(low_ss_auc_high_data)} species")
print(f"  → Action: Overfitting clean recordings. Increase 'Background Noise' augmentation.")

if not high_gap_species.empty:
    print(f"\nHigh Domain Shift (Gap > {HIGH_GAP_THRESH})              : {len(high_gap_species)} species")
    print(f"  → Action: Clean-data specialists. Apply 'Gaussian Noise' and 'Gain' during training.")

# Class-level check for soundscapes
for cls in ss_auc_df['class_name'].unique():
    if pd.isna(cls): continue
    cls_ss_mean = ss_auc_df[ss_auc_df['class_name'] == cls]['ss_auc'].mean()
    print(f"\n{cls} Mean Soundscape AUC                       : {cls_ss_mean:.4f}")

print("\n" + "=" * 75)
print(f"OVERALL MEAN SOUNDSCAPE AUC: {ss_auc_df['ss_auc'].mean():.4f}")
if 'auc_gap' in ss_auc_df.columns:
    print(f"OVERALL MEAN AUC GAP (TRAIN vs SS): {ss_auc_df['auc_gap'].mean():.4f}")
print("=" * 75)

print("\nPRIORITY SPECIES: High Domain Shift (Learn in Train, Fail in Soundscape)")
if not high_gap_species.empty:
    cols = ['common_name', 'ss_auc', 'train_auc', 'auc_gap', 'train_count']
    print(high_gap_species[cols].sort_values('auc_gap', ascending=False).head(15).to_string())
else:
    print("No high-gap species detected.")

SOUNDSCAPE ACTIONABLE SUMMARY
Low SS-AUC (<0.7) + Low data (<50)      : 0 species
  → Action: High priority for Pseudo-labeling. Model needs field context.

Low SS-AUC (<0.7) + Enough data (≥50)   : 0 species
  → Action: Overfitting clean recordings. Increase 'Background Noise' augmentation.

Reptilia Mean Soundscape AUC                       : 0.9889

Aves Mean Soundscape AUC                       : 0.9984

Amphibia Mean Soundscape AUC                       : 0.9993

Insecta Mean Soundscape AUC                       : 0.9997

Mammalia Mean Soundscape AUC                       : 0.9995

OVERALL MEAN SOUNDSCAPE AUC: 0.9990
OVERALL MEAN AUC GAP (TRAIN vs SS): -0.0065

PRIORITY SPECIES: High Domain Shift (Learn in Train, Fail in Soundscape)
No high-gap species detected.


In [21]:
# ====================================================
# SITE-LEVEL ANALYSIS OF WORST PERFORMING BIRDS
# ====================================================

# 1. Identify the 'Worst' birds from our previous Soundscape AUC analysis
# We'll take the bottom 20 species by ss_auc
worst_birds_list = ss_auc_df.head(20).index.tolist()

# 2. Extract Site IDs and map occurrences of these worst birds
site_stats = []

for _, row in ss_df.iterrows():
    # Extract Site ID (e.g., 'S22' from 'BC2026_Train_0039_S22_...')
    try:
        site = row['filename'].split('_')[3]
    except:
        site = 'Unknown'
        
    # Check if any of the 'worst' birds are in this segment
    for bird in row['bird_list']:
        if bird in worst_birds_list:
            site_stats.append({
                'bird': bird,
                'site': site,
                'common_name': ss_auc_df.loc[bird, 'common_name'] if bird in ss_auc_df.index else bird
            })

site_df = pd.DataFrame(site_stats)

if not site_df.empty:
    print("=" * 70)
    print("TOP SITES WHERE WORST-PERFORMING BIRDS OCCUR")
    print("=" * 70)
    site_counts = site_df['site'].value_counts()
    print(site_counts.to_string())
    
    print("\n" + "=" * 70)
    print("SITE BREAKDOWN BY SPECIES (Top 10 pairings)")
    print("=" * 70)
    top_pairings = site_df.groupby(['site', 'common_name']).size().reset_index(name='count')
    print(top_pairings.sort_values('count', ascending=False).head(10).to_string(index=False))
else:
    print("No occurrences of worst-performing birds found in the site data.")

TOP SITES WHERE WORST-PERFORMING BIRDS OCCUR
site
S22    598
S15     67
S23     43
S08     41
S03     36
S13     20
S19     12
S18      5
S09      2

SITE BREAKDOWN BY SPECIES (Top 10 pairings)
site                      common_name  count
 S22 Southern Orange-legged Leaf Frog    269
 S22         Paraguayan Swimming Frog    149
 S22         Guaraní leaf-litter frog     53
 S22                         Pauraque     37
 S15                 Chaco Chachalaca     32
 S22                Undulated Tinamou     31
 S22             Tropical Screech Owl     26
 S23                Insect sonotype10     25
 S03 Southern Orange-legged Leaf Frog     22
 S08                 Chaco Chachalaca     17


In [22]:
# ====================================================
# SITE-LEVEL ERROR ANALYSIS: Correct vs Wrong
# ====================================================

# We'll use a threshold of 0.3 here, as soundscapes are noisy 
# and 0.5 is often too restrictive for mAP optimization.
DETECTION_THRESHOLD = 0.3

site_error_data = []

for i, row in ss_df.iterrows():
    # Extract Site ID
    try:
        site = row['filename'].split('_')[3]
    except:
        site = 'Unknown'
        
    gt_species = set(row['bird_list'])
    
    # Get predictions based on threshold
    top_k = 5
    pred_indices = np.argsort(ss_probs[i])[-top_k:]
    pred_species = set([all_species_codes[idx] for idx in pred_indices])
    
    # 1. Correct Detections (True Positives)
    for s in gt_species.intersection(pred_species):
        site_error_data.append({'site': site, 'species': s, 'result': 'Correct (GT)'})
        
    # 2. Hallucinations (False Positives) - Predicted but NOT in GT
    for s in pred_species - gt_species:
        site_error_data.append({'site': site, 'species': s, 'result': 'Wrong (Hallucination)'})
        
    # 3. Misses (False Negatives) - In GT but NOT predicted
    for s in gt_species - pred_species:
        site_error_data.append({'site': site, 'species': s, 'result': 'Missed (In Ground Truth)'})

errors_df = pd.DataFrame(site_error_data)

# Map species codes to common names for readability
errors_df['common_name'] = errors_df['species'].map(
    taxonomy_df.set_index('primary_label')['common_name']
).fillna(errors_df['species'])

# Print Summary for each Site
for site in sorted(errors_df['site'].unique()):
    site_data = errors_df[errors_df['site'] == site]
    
    print(f"\n{'='*75}")
    print(f"SITE: {site} (Total Segments Analyzed: {len(ss_df[ss_df['filename'].str.contains(site)])})")
    print(f"{'='*75}")
    
    # Top Correct
    correct = site_data[site_data['result'] == 'Correct (GT)']
    print(f"\n[+] TOP CORRECT SPECIES (TP):")
    if not correct.empty:
        print(correct['common_name'].value_counts().head(5).to_string())
    else:
        print("None")
        
    # Top Hallucinations
    hallucs = site_data[site_data['result'] == 'Wrong (Hallucination)']
    print(f"\n[!] TOP HALLUCINATIONS (FP - Model heard it, but it's not there):")
    if not hallucs.empty:
        print(hallucs['common_name'].value_counts().head(10).to_string())
    else:
        print("None")

    # Top Misses
    misses = site_data[site_data['result'] == 'Missed (In Ground Truth)']
    print(f"\n[-] TOP MISSED SPECIES (FN - Bird sang, but model was deaf to it):")
    if not misses.empty:
        print(misses['common_name'].value_counts().head(5).to_string())
    else:
        print("None")


SITE: S03 (Total Segments Analyzed: 24)

[+] TOP CORRECT SPECIES (TP):
common_name
Southern Orange-legged Leaf Frog    22
Guaraní leaf-litter frog            14
Whistling Grass Frog                12
Marbled White-lipped Frog           12

[!] TOP HALLUCINATIONS (FP - Model heard it, but it's not there):
common_name
Tropical Screech Owl                12
Lesser Snouted Tree Frog             8
Undulated Tinamou                    8
Marbled White-lipped Frog            6
Guaraní leaf-litter frog             5
Whistling Grass Frog                 5
Dwarf Tree Frog                      5
Little Nightjar                      2
Southern Orange-legged Leaf Frog     2
Chaco Tree Frog                      2

[-] TOP MISSED SPECIES (FN - Bird sang, but model was deaf to it):
None

SITE: S08 (Total Segments Analyzed: 60)

[+] TOP CORRECT SPECIES (TP):
common_name
Insect sonotype25    44
Insect sonotype17    42
Insect sonotype22    24
Insect sonotype23    24
Insect sonotype13    24

[!] TOP HALLU

In [ ]:
# =========================
# AMPHIBIAN-SPECIFIC SUMMARY
# Directly answers: will an amphibian specialist help?
# =========================
amphibian_ss = ss_auc_df[ss_auc_df['class_name'] == 'Amphibia'].copy()
amphibian_train = auc_df[auc_df['class_name'] == 'Amphibia'].copy()

print("=" * 75)
print("AMPHIBIAN PERFORMANCE SUMMARY")
print("=" * 75)
print(f"Amphibian species in soundscapes : {len(amphibian_ss)}")
print(f"Mean soundscape AUC (Amphibia)   : {amphibian_ss['ss_auc'].mean():.4f}")
print(f"Mean train AUC (Amphibia)        : {amphibian_train['auc'].mean():.4f}")
print(f"Mean domain gap (Amphibia)       : {amphibian_ss['auc_gap'].mean():.4f}")
print()

# Species below 0.7 soundscape AUC — these are where specialist helps
weak_amphibians = amphibian_ss[amphibian_ss['ss_auc'] < 0.70]
print(f"Weak amphibian species (ss_auc < 0.70): {len(weak_amphibians)}")
print()
print(weak_amphibians[['common_name', 'ss_auc', 'train_auc', 
                         'auc_gap', 'train_count']].sort_values('ss_auc').to_string())

# Decision rule
print()
print("=" * 75)
print("SPECIALIST DECISION")
print("=" * 75)
mean_amp_ss = amphibian_ss['ss_auc'].mean()
weak_count  = len(weak_amphibians)

if mean_amp_ss > 0.80 and weak_count < 5:
    print(f"B3 already strong on amphibians (mean={mean_amp_ss:.3f}, weak={weak_count})")
    print("→ Specialist unlikely to move the ensemble. Skip or weight very low (0.05).")
elif weak_count >= 10:
    print(f"B3 struggling on {weak_count} amphibian species (mean={mean_amp_ss:.3f})")
    print("→ Specialist is WORTH training. Weight it meaningfully (0.15-0.20) in ensemble.")
else:
    print(f"Mixed picture (mean={mean_amp_ss:.3f}, weak={weak_count})")
    print("→ Train specialist but weight conservatively (0.10) in ensemble.")

AMPHIBIAN PERFORMANCE SUMMARY
Amphibian species in soundscapes : 17
Mean soundscape AUC (Amphibia)   : 0.9993
Mean train AUC (Amphibia)        : 0.9982
Mean domain gap (Amphibia)       : -0.0033

Weak amphibian species (ss_auc < 0.70): 0

Empty DataFrame
Columns: [common_name, ss_auc, train_auc, auc_gap, train_count]
Index: []

SPECIALIST DECISION
B3 already strong on amphibians (mean=0.999, weak=0)
→ Specialist unlikely to move the ensemble. Skip or weight very low (0.05).
